# DES Y3 vs fiducial-mock Cl discriminator (WL audit §6.1)

**Test (`wl_forward_model_audit_2026-07-11.md`, §6 item 1).** The leading remaining hypothesis for
the WL constraining-power gap is a *mild data-vs-sim mismatch in fine Cl features* (§4.3, §5.3
hyp. 3). This notebook runs the cheapest, most direct test of it: no compression is needed for the
2-point path, so we compare the **real DES Y3 hard-rebinned Cl vector** directly against the
**39 960-realization fiducial mock ensemble** in `cls/fiducial_cls.h5`.

We compute, per probe (lensing / clustering / combined):
- per-column **z-score** `z = (d_DES - mu) / s`,
- a whitened **chi^2** (Hartlap-debiased) with an analytic PTE **and** an empirical PTE from the mock
  chi^2 distribution,
- the chi^2 **localized** per bin-pair and per ell-bin,

and a z-score heatmap over (bin-pair, ell-bin). If the mismatch is real it must appear here, and
*where* it sits discriminates the candidates:

| localization | candidate |
|---|---|
| high-ell lensing **auto** plateaus | noise-depth mismatch |
| **low-z** bins at **mid-ell** | baryon / IA residuals |
| **low ell** | additive systematics |

Comparison is against the **fiducial** ensemble only: the DES data sits at its own best-fit
cosmology, so the residual mixes the smooth `~ dCl/dtheta` cosmology offset with the mismatch — but a
cosmology offset is coherent across ell (it follows the derivative shape), whereas a mismatch is
*localized*, which is exactly what the ell/bin resolution separates.

**Run on a compute node in the TensorFlow env** (`srun --environment=tensorflow`, then
`source ~/dlss/tf_env/bin/activate`): it builds the DES maps from the real catalogs and imports the
msfm / deep_lss stack. No training, no GPU.

In [ ]:
import os
import yaml

import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy import stats

from msfm.utils import files, cross_statistics
from deep_lss.utils.cls_preprocessing import (
    _build_bin_weights_all_pairs,
    preprocess_obs_hard_rebinned,
    get_rebinned_pair_info,
)

REPO = "/users/athomsen/dlss/repos"
MSFM_CONFIG = f"{REPO}/multiprobe-simulation-forward-model/configs/v17/baseline.yaml"
SCALES_CONFIG = f"{REPO}/y3-deep-lss/configs/scales/8wl,32gc.yaml"  # production cut
FIDUCIAL_H5 = "/users/athomsen/scratch/deep_lss/data/v17/baseline/cls/fiducial_cls.h5"
OUT_DIR = "/iopsstor/scratch/cscs/athomsen/deep_lss/runs/wl_audit/cl_discriminator"
os.makedirs(OUT_DIR, exist_ok=True)

CLS_N_BINS = 16

# configs as dicts (files.load_config is idempotent on dicts; catalog/preprocess accept dicts)
msfm_conf = files.load_config(MSFM_CONFIG)
with open(SCALES_CONFIG) as f:
    dlss_conf = yaml.safe_load(f)  # top-level key "scale_cuts"

n_z_lensing = len(msfm_conf["survey"]["metacal"]["z_bins"])
n_z_clustering = len(msfm_conf["survey"]["maglim"]["z_bins"])
n_side = msfm_conf["analysis"]["n_side"]
n_ell = 3 * n_side
print(f"n_z_lensing={n_z_lensing} n_z_clustering={n_z_clustering} n_side={n_side} n_ell={n_ell}")

## 1. Scale cut + hard-rebin weight tensor

The 16 sqrt-spaced ell-bins per pair with the per-pair `8wl,32gc` cut, built by the exact training
utility `deep_lss.utils.cls_preprocessing._build_bin_weights_all_pairs` (so the ensemble and the DES
vector are rebinned identically).

In [ ]:
sc = dlss_conf["scale_cuts"]
l_min_z = list(sc["lensing"]["l_min"]) + list(sc["clustering"]["l_min"])
l_max_z = list(sc["lensing"]["l_max"]) + list(sc["clustering"]["l_max"])
assert len(l_min_z) == len(l_max_z) == n_z_lensing + n_z_clustering
print("l_min_z =", l_min_z)
print("l_max_z =", l_max_z)

W = _build_bin_weights_all_pairs(n_ell, CLS_N_BINS, n_z_lensing, n_z_clustering, l_min_z, l_max_z)
print("W", W.shape)  # (1536, 16, 36)

## 2. Fiducial mock ensemble -> rebinned realizations

Read `cls/raw` `(39960, 1536, 36)` (3996 signal x 10 noise) and rebin to `(39960, 16, 36)`, chunked
to bound memory. Same einsum as `fisher_cls.compute_rebinned_covariance`.

In [ ]:
def rebin_raw(fiducial_h5, W, chunk=4000):
    with h5py.File(fiducial_h5, "r") as f:
        raw = f["cls/raw"]
        n, nl, npair = raw.shape
        assert (nl, npair) == (W.shape[0], W.shape[2]), (raw.shape, W.shape)
        Xb = np.empty((n, W.shape[1], npair), dtype=np.float64)
        for i0 in range(0, n, chunk):
            i1 = min(i0 + chunk, n)
            Xb[i0:i1] = np.einsum("nlc,lkc->nkc", raw[i0:i1].astype(np.float64), W, optimize=True)
    return Xb


Xb = rebin_raw(FIDUCIAL_H5, W)
print("Xb", Xb.shape)  # (39960, 16, 36)

## 3. DES Y3 observed Cl vector

Build the metacal (shear) and maglim (counts) maps from the real DES catalogs, then run the *same*
`preprocess_obs_hard_rebinned(..., apply_log=False)` used for the network's obs vector (linear Cls,
not the sign-log network input). Mirrors `msi.utils.ppc._build_cls_obs` /
`deep_lss.utils.cls_evaluation.evaluate_des_y3`. Maps are built once (both probes) and reused.

In [ ]:
from msfm.utils import catalog

print("Building DES Y3 maps from catalogs (this is the slow step)...")
wl_gamma_map, _ = catalog.build_metacal_map_from_cat(msfm_conf)
gc_count_map = catalog.build_maglim_map_from_cat(msfm_conf)
print("wl_gamma_map", np.shape(wl_gamma_map), " gc_count_map", np.shape(gc_count_map))

## 4. Whitening + discriminator statistics per probe

For each probe we select its columns with `get_cross_bin_indices` (36 pairs total; lensing bins 0-3,
clustering 4-7), flatten bin-major/pair-minor (`col = ell_bin * n_sel + pair`, matching the obs
preprocessing), then:
- `mu = mean`, `C = cov`, `s = sqrt(diag C)`, correlation matrix `R = C / (s s^T)`;
- whitened residual `r = (d_DES - mu)/s`; whitened chi^2 `= h * r^T R^{-1} r` (`h` = Hartlap), dof `= d`;
- empirical PTE = fraction of mocks with chi^2 >= the DES chi^2 (distribution-free, uses all 39 960);
- localized chi^2 per pair (its own 16-col R sub-block) and per ell-bin (across pairs).

Inverting the well-scaled correlation matrix `R` (not `C`, which spans ~14 orders of magnitude) keeps
the solve numerically stable, exactly as in `fisher_cls.fisher_forecast`.

In [ ]:
PROBE_FLAGS = {
    "lensing":    dict(with_lensing=True,  with_clustering=False, with_cross_z=True, with_cross_probe=False),
    "clustering": dict(with_lensing=False, with_clustering=True,  with_cross_z=True, with_cross_probe=False),
    "combined":   dict(with_lensing=True,  with_clustering=True,  with_cross_z=True, with_cross_probe=True),
}


def flatten_pairs(arr, bin_indices):
    # Select pairs and flatten bin-major/pair-minor: col = ell_bin * n_sel + pair.
    # Identical order to cls_preprocessing.preprocess_obs_hard_rebinned.
    sub = arr[..., :, bin_indices]
    return sub.reshape(sub.shape[:-2] + (-1,))


def hartlap(n_sim, d):
    return (n_sim - d - 2.0) / (n_sim - 1.0)


results = {}
for probe, flags in PROBE_FLAGS.items():
    bin_indices, _ = cross_statistics.get_cross_bin_indices(
        n_z_lensing=n_z_lensing, n_z_clustering=n_z_clustering, **flags
    )
    bin_indices = list(bin_indices)
    n_sel = len(bin_indices)

    # ensemble
    X = flatten_pairs(Xb, bin_indices)  # (N, d)
    N, d = X.shape
    mu = X.mean(axis=0)
    C = np.cov(X, rowvar=False)
    s = np.sqrt(np.diag(C))
    R = C / np.outer(s, s)
    np.linalg.cholesky(R)  # PD check on the well-scaled correlation matrix
    Rinv = np.linalg.inv(R)
    h = hartlap(N, d)
    cond_C, cond_R = np.linalg.cond(C), np.linalg.cond(R)

    # DES vector, identical rebin + probe selection
    d_des = (
        preprocess_obs_hard_rebinned(
            wl_gamma_map=wl_gamma_map,
            gc_count_map=gc_count_map,
            msfm_conf=msfm_conf,
            dlss_conf=dlss_conf,
            cls_n_bins=CLS_N_BINS,
            apply_log=False,
            **flags,
        )
        .reshape(-1)
        .astype(np.float64)
    )
    assert d_des.shape == mu.shape, (d_des.shape, mu.shape)

    # z-scores + whitened chi^2 (Hartlap-debiased)
    z = (d_des - mu) / s
    chi2 = float(h * (z @ Rinv @ z))
    pte_analytic = float(stats.chi2.sf(chi2, df=d))

    # empirical PTE from the mock chi^2 distribution (in-sample mean/cov; a reference dist)
    Zw = (X - mu) / s  # (N, d)
    chi2_mocks = h * np.einsum("nd,nd->n", Zw @ Rinv, Zw)
    pte_empirical = float((chi2_mocks >= chi2).mean())

    # localized chi^2 per pair (16 cols each) and per ell-bin (n_sel cols each)
    per_pair_chi2 = np.empty(n_sel)
    per_pair_pte = np.empty(n_sel)
    for pp in range(n_sel):
        cols = np.array([b * n_sel + pp for b in range(CLS_N_BINS)])
        rsub = z[cols]
        Rsub_inv = np.linalg.inv(R[np.ix_(cols, cols)])
        hsub = hartlap(N, len(cols))
        c2 = float(hsub * (rsub @ Rsub_inv @ rsub))
        per_pair_chi2[pp] = c2
        per_pair_pte[pp] = float(stats.chi2.sf(c2, df=len(cols)))

    per_ell_chi2 = np.empty(CLS_N_BINS)
    per_ell_pte = np.empty(CLS_N_BINS)
    for b in range(CLS_N_BINS):
        cols = np.array([b * n_sel + pp for pp in range(n_sel)])
        rsub = z[cols]
        Rsub_inv = np.linalg.inv(R[np.ix_(cols, cols)])
        hsub = hartlap(N, len(cols))
        c2 = float(hsub * (rsub @ Rsub_inv @ rsub))
        per_ell_chi2[b] = c2
        per_ell_pte[b] = float(stats.chi2.sf(c2, df=len(cols)))

    labels, ell_centers, ell_ranges = get_rebinned_pair_info(
        msfm_conf, dlss_conf, CLS_N_BINS, **flags
    )

    results[probe] = dict(
        n_sel=n_sel, d=d, N=N, h=h, cond_C=cond_C, cond_R=cond_R,
        z=z, chi2=chi2, pte_analytic=pte_analytic, pte_empirical=pte_empirical,
        chi2_mocks=chi2_mocks, per_pair_chi2=per_pair_chi2, per_pair_pte=per_pair_pte,
        per_ell_chi2=per_ell_chi2, per_ell_pte=per_ell_pte,
        labels=labels, ell_centers=ell_centers, ell_ranges=ell_ranges,
    )
    print(
        f"[{probe:10s}] d={d:4d} n_sel={n_sel:2d} cond(C)={cond_C:.1e} cond(R)={cond_R:.1e} "
        f"hartlap={h:.4f} | chi2={chi2:.1f}/{d} PTE_a={pte_analytic:.3g} PTE_e={pte_empirical:.3g}"
    )

## 5. Sanity checks

Before trusting the DES numbers, verify the whitening + Hartlap on the mocks themselves: the mock
self-chi^2 median should sit near the dof, and the per-mock analytic PTEs should be ~Uniform(0,1)
(the ensemble is Gaussian-ish by construction). Column dimensions: lensing/clustering `d = 160`
(10 pairs x 16), combined `d = 576` (36 x 16).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)
for ax, probe in zip(axes, PROBE_FLAGS):
    r = results[probe]
    d, N, h = r["d"], r["N"], r["h"]
    med = np.median(r["chi2_mocks"])
    ax.hist(r["chi2_mocks"], bins=80, density=True, color="0.7", edgecolor="none")
    xs = np.linspace(r["chi2_mocks"].min(), r["chi2_mocks"].max(), 400)
    ax.plot(xs, stats.chi2.pdf(xs, df=d), color="C0", lw=2, label=f"chi2(dof={d})")
    ax.axvline(med, color="0.3", ls="--", lw=1.5, label=f"mock median={med:.0f}")
    ax.axvline(r["chi2"], color="C3", lw=2, label=f"DES={r['chi2']:.0f}")
    ax.set_title(probe)
    ax.set_xlabel("whitened chi^2")
    ax.legend(fontsize=8)
    print(f"[{probe:10s}] mock self-chi2 median={med:.1f} (dof={d}); DES chi2={r['chi2']:.1f}")
axes[0].set_ylabel("density")
fig.suptitle("Mock self-chi^2 distribution vs analytic dof (sanity) + DES value")
fig.savefig(f"{OUT_DIR}/sanity_chi2.png", dpi=130)
plt.show()

## 6. z-score heatmaps (localized in bin-pair x ell-bin)

Diverging map centered at 0 (RdBu_r; neutral-gray midpoint per diverging-palette rule), symmetric
limits. Rows = tomographic pairs in data-vector order (kappa^i for lensing, delta_g^i for
clustering), annotated with each pair's [l_min, l_max]; columns = the 16 sqrt-spaced ell-bins. A
smooth block across ell within a pair is a cosmology-like offset; an isolated hot cell is a localized
mismatch.

In [ ]:
def heatmap(probe, ax):
    r = results[probe]
    n_sel, z = r["n_sel"], r["z"]
    Zmat = z.reshape(CLS_N_BINS, n_sel).T  # (n_sel, 16): rows=pair, cols=ell-bin
    vmax = np.nanpercentile(np.abs(Zmat), 99)
    im = ax.imshow(Zmat, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(CLS_N_BINS))
    ax.set_xticklabels(range(CLS_N_BINS), fontsize=7)
    ax.set_xlabel("ell-bin index (sqrt-spaced, low -> high ell)")
    row_labels = [f"{lab}  [{int(lo)},{int(hi)}]" for lab, (lo, hi) in zip(r["labels"], r["ell_ranges"])]
    ax.set_yticks(range(n_sel))
    ax.set_yticklabels(row_labels, fontsize=7)
    ax.set_title(f"{probe}: DES z-score  (chi2={r['chi2']:.0f}/{r['d']}, PTE_emp={r['pte_empirical']:.3g})")
    plt.colorbar(im, ax=ax, label="z = (DES - mock mean)/sigma", shrink=0.8)


for probe in PROBE_FLAGS:
    h_in = max(3.0, 0.32 * results[probe]["n_sel"])
    fig, ax = plt.subplots(figsize=(9, h_in), constrained_layout=True)
    heatmap(probe, ax)
    fig.savefig(f"{OUT_DIR}/zscore_heatmap_{probe}.png", dpi=130)
    plt.show()

## 7. Localized chi^2 per pair and per ell-bin

Where does the departure concentrate? The per-pair panel isolates which tomographic combinations
drive it (baryon/IA -> low-z lensing pairs); the per-ell panel isolates the scale (additive sys ->
low ell; noise depth -> high ell). Bars are colored by PTE significance (a reserved status ramp, not
a series color).

In [ ]:
def pte_colors(ptes):
    out = []
    for p in ptes:
        if p < 0.003:
            out.append("#b2182b")   # critical  (~3 sigma)
        elif p < 0.05:
            out.append("#ef8a62")   # serious
        else:
            out.append("#9ecae1")   # ok
    return out


for probe in PROBE_FLAGS:
    r = results[probe]
    n_sel = r["n_sel"]
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(10, 6.5), constrained_layout=True)

    a0.bar(range(n_sel), r["per_pair_chi2"], color=pte_colors(r["per_pair_pte"]))
    a0.axhline(CLS_N_BINS, color="0.4", ls="--", lw=1, label=f"dof={CLS_N_BINS}")
    a0.set_xticks(range(n_sel))
    a0.set_xticklabels(r["labels"], rotation=90, fontsize=7)
    a0.set_ylabel("chi^2 (16 ell-bins)")
    a0.set_title(f"{probe}: per-pair whitened chi^2 (red=PTE<0.003, orange=<0.05)")
    a0.legend(fontsize=8)

    a1.bar(range(CLS_N_BINS), r["per_ell_chi2"], color=pte_colors(r["per_ell_pte"]))
    a1.axhline(n_sel, color="0.4", ls="--", lw=1, label=f"dof={n_sel}")
    a1.set_xticks(range(CLS_N_BINS))
    a1.set_xlabel("ell-bin index (sqrt-spaced, low -> high ell)")
    a1.set_ylabel(f"chi^2 ({n_sel} pairs)")
    a1.set_title(f"{probe}: per-ell-bin whitened chi^2")
    a1.legend(fontsize=8)

    fig.savefig(f"{OUT_DIR}/localized_chi2_{probe}.png", dpi=130)
    plt.show()

## 8. Save arrays + summary

Persist the z-scores / chi^2 / PTEs to scratch for the audit-note writeup (home VAST quota silently
drops large writes — outputs go to `/iopsstor` scratch).

In [ ]:
save = {}
for probe, r in results.items():
    for k in ["z", "chi2", "pte_analytic", "pte_empirical", "per_pair_chi2", "per_pair_pte",
              "per_ell_chi2", "per_ell_pte", "ell_ranges", "d", "n_sel", "N", "h"]:
        save[f"{probe}/{k}"] = np.asarray(r[k])
    save[f"{probe}/labels"] = np.array(r["labels"], dtype=object)
np.savez(f"{OUT_DIR}/cl_discriminator.npz", **save)
print("saved ->", f"{OUT_DIR}/cl_discriminator.npz")

print("\n=== DES Y3 vs fiducial ensemble: consistency summary ===")
print(f"{'probe':11s} {'dof':>4s} {'chi2':>9s} {'chi2/dof':>9s} {'PTE_analytic':>13s} {'PTE_empirical':>14s}")
for probe, r in results.items():
    print(f"{probe:11s} {r['d']:>4d} {r['chi2']:>9.1f} {r['chi2']/r['d']:>9.3f} "
          f"{r['pte_analytic']:>13.3g} {r['pte_empirical']:>14.3g}")

## 9. Interpretation

Read the panels against the §6.1 candidate map:

- **Consistent (PTE not small, no localized hot cells):** the mismatch is *not* a Gaussian 2-point
  data-vector feature. That pushes hypothesis 3 toward the map-level / non-Gaussian channel and
  motivates the §6 item 2 summary-direction attribution (which Cl features the extra summary
  directions read) rather than a raw-Cl defect.
- **High-ell lensing-auto excess:** shape-noise / depth modelling (uniform sim noise depth).
- **Low-z lensing pairs at mid-ell:** baryon or IA model residuals.
- **Low-ell excess (any probe):** additive systematics (PSF residual, depth, stellar density) — the
  §6 item 3 systematics-template injection is the follow-up.
- **Clustering-dominated:** linear-bias / n(z) modelling; note clustering is capped at ~32 Mpc/h by
  design, so high-ell clustering bins are not part of the production vector.

Record the per-probe PTE and any localized excess in
`multiprobe-simulation-forward-model/dev/notes/wl_forward_model_audit_2026-07-11.md` and flip §6
item 1 to done.